In [ ]:
!pip install -q datasets transformers torchaudio soundfile pandas jiwer accelerate evaluate scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 117.4 MB/s eta 0:00:00


In [ ]:
# ============================================================
# 1  Imports
# ============================================================

import os
import re
import json
import torch
import pandas as pd
import numpy as np

from dataclasses import dataclass
from typing import Union

from datasets import Dataset, Audio
from evaluate import load
from sklearn.model_selection import train_test_split
from transformers import set_seed

import transformers
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,          # MMS uses Wav2Vec2ForCTC (not Conformer)
    TrainingArguments,
    Trainer,
)

print("Transformers version:", transformers.__version__)
set_seed(42)

# ============================================================
# 2  Paths  (edit if your layout differs)
# ============================================================

DATA_DIR   = "/content/zambezi-voice-nyanja/nya/"
AUDIO_DIR  = os.path.join(DATA_DIR, "audio/")
TRAIN_TSV  = os.path.join("/content/zambezi-voice-nyanja/nya/train.tsv")
OUTPUT_DIR = "./nyanja-mms-model"
FINAL_DIR  = "./nyamja-mms-final"
VOCAB_FILE = "vocab.json"


Transformers version: 5.0.0


In [ ]:
pd.read_csv(TRAIN_TSV, sep=",").head()

,\taudio_id\tsentence\tBitsPerSample\tdurationMsec\tsampleRate
0,0\t221102-102320_nya_510_elicit_0.wav\twosewer...
1,1\t221102-102320_nya_510_elicit_1.wav\tmunthu ...
2,2\t221102-102320_nya_510_elicit_2.wav\tmwana w...
3,3\t221102-102320_nya_510_elicit_3.wav\twosewer...
4,4\t221102-102320_nya_510_elicit_4.wav\tana ata...


In [ ]:

# ============================================================
# 3  Load Dataset
# ============================================================

print("Loading nyanja dataset...")

df = pd.read_csv(TRAIN_TSV, sep="\t")
df = df[["audio_id", "sentence"]].rename(columns={"sentence": "text"})
df["path"] = AUDIO_DIR + df["audio_id"]
df = df.dropna()

total_before = len(df)
df["exists"]   = df["path"].apply(os.path.exists)
found_df       = df[df["exists"]]
missing_df     = df[~df["exists"]]

print(f"Total rows in TSV (after dropna): {total_before}")
print(f"Audio files FOUND:   {len(found_df)}")
print(f"Audio files MISSING: {len(missing_df)}")

if len(missing_df) > 0:
    print(f"Missing rate: {len(missing_df)/total_before*100:.1f}%")
    for _, row in missing_df.head(10).iterrows():
        print(f"  {row['path']}")

df = df[df["exists"]][["path", "text"]].reset_index(drop=True)
print(f"\nUsable samples: {len(df)}")


Loading nyanja dataset...
Total rows in TSV (after dropna): 8117
Audio files FOUND:   8117
Audio files MISSING: 0

Usable samples: 8117


In [ ]:
# ============================================================
# 4  Train / Validation Split
# ============================================================

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))

# ============================================================
# 5  HuggingFace Dataset
# ============================================================

def create_dataset(df):
    dataset = Dataset.from_pandas(df)
    dataset = dataset.cast_column("path", Audio(sampling_rate=16000))
    return dataset

train_dataset = create_dataset(train_df)
val_dataset   = create_dataset(val_df)

# ============================================================
# 6  Clean Text  (Nyanja-specific)
# ============================================================

CHARS_TO_REMOVE   = r'[,?.!-\;:""%\'\"\'\"\'…()[\]{}]'
NON_NYANJA_LETTERS = r'[qx]'   #Tonga does NOT use q or x

def clean_text(batch):
    text = batch["text"].lower()
    text = re.sub(CHARS_TO_REMOVE, "", text)
    text = re.sub(r"[0-9]", "", text)
    text = re.sub(NON_NYANJA_LETTERS, "", text)
    text = re.sub(r"[^\x00-\x7F]", "", text)   # strip non-ASCII
    text = re.sub(r" +", " ", text).strip()
    batch["text"] = text
    return batch

train_dataset = train_dataset.map(clean_text)
val_dataset   = val_dataset.map(clean_text)

train_dataset = train_dataset.filter(lambda x: len(x["text"].strip()) > 0)
val_dataset   = val_dataset.filter(lambda x: len(x["text"].strip()) > 0)

print("\n--- Sample cleaned text ---")
for i in range(5):
    print(repr(train_dataset[i]["text"]))
print("---\n")

# ============================================================
# 7  Build Vocabulary  (Nyanja-specific, 27 chars)
# ============================================================

def extract_chars(batch):
    all_text = " ".join(batch["text"])
    return {"vocab": [list(set(all_text))]}

vocab_dataset = train_dataset.map(
    extract_chars,
    batched=True,
    batch_size=-1,
    remove_columns=train_dataset.column_names,
)

vocab_list = sorted(set(vocab_dataset["vocab"][0]))
vocab_dict = {v: i for i, v in enumerate(vocab_list)}

if " " in vocab_dict:
    vocab_dict["|"] = vocab_dict[" "]
    del vocab_dict[" "]

for bad_char in ["q", "x"]:
    vocab_dict.pop(bad_char, None)

vocab_dict = {char: idx for idx, char in enumerate(sorted(vocab_dict.keys()))}
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print("Vocabulary size:", len(vocab_dict))
print("Vocabulary:", sorted(vocab_dict.keys()))

assert "q" not in vocab_dict, "ERROR: 'q' found in vocabulary!"
assert "x" not in vocab_dict, "ERROR: 'x' found in vocabulary!"
print("✓ Confirmed: 'q' and 'x' are NOT in vocabulary")

with open(VOCAB_FILE, "w") as f:
    json.dump(vocab_dict, f)

# ============================================================
# 8  Processor
# ============================================================

tokenizer = Wav2Vec2CTCTokenizer(
    VOCAB_FILE,
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

# ============================================================
# 9  Prepare Dataset
# ============================================================

def prepare_dataset(batch):
    audio = batch["path"]
    batch["input_values"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    ).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names)
val_dataset   = val_dataset.map(prepare_dataset,   remove_columns=val_dataset.column_names)

# ============================================================
# 10  Filter Audio  (1 s – 30 s)
# ============================================================

MIN_LEN = 1  * 16_000
MAX_LEN = 30 * 16_000

train_dataset = train_dataset.filter(lambda x: MIN_LEN < x["input_length"] < MAX_LEN)
val_dataset   = val_dataset.filter(  lambda x: MIN_LEN < x["input_length"] < MAX_LEN)

print("Train after filtering:", len(train_dataset))
print("Validation after filtering:", len(val_dataset))

# ============================================================
# 11  Data Collator
# ============================================================

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]}          for f in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor)


Train: 7305
Validation: 812


Map:   0%|          | 0/7305 [00:00<?, ? examples/s]

Map:   0%|          | 0/812 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7305 [00:00<?, ? examples/s]

Filter:   0%|          | 0/812 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/7305 [00:00<?, ? examples/s]

Map:   0%|          | 0/812 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7305 [00:00<?, ? examples/s]

Filter:   0%|          | 0/812 [00:00<?, ? examples/s]

Train after filtering: 7290
Validation after filtering: 811


In [ ]:
# ============================================================
# 12  Load MMS Model
#
#     facebook/mms-300m
#     - Meta Massively Multilingual Speech model
#     - Pre-trained on 1000+ languages including many Bantu
#       languages
#     - ignore_mismatched_sizes=True drops the MMS vocab head
#       and initialises a fresh Nyanja head (27 chars)
#
#     Other MMS options if you want to experiment:
#       facebook/mms-1b          (1B params, best quality, slow)
#       facebook/mms-1b-fl102    (fine-tuned on 102 languages)
#       facebook/mms-1b-all      (fine-tuned on 1162 languages)
# ============================================================

MODEL_NAME = "facebook/mms-300m"

model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.1,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True,   # drops MMS vocab head, adds fresh Nyanja head
)

model.freeze_feature_encoder()
print(f"Model loaded: {MODEL_NAME}")
print(f"Total parameters: {model.num_parameters():,}")

# ============================================================
# 13  Training Arguments
#     Lower LR (1e-4) and more epochs (5)
#     for MMS fine-tuning on low-resource languages
# ============================================================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=100,
    save_steps=200,
    save_strategy="steps",
    num_train_epochs=5,        # MMS benefits from more epochs
    learning_rate=1e-4,         # lower LR suits MMS fine-tuning
    warmup_steps=500,
    weight_decay=0.005,
    gradient_checkpointing=True,
    fp16=True,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
    length_column_name="input_length",
)

# ============================================================
# 14  WER Metric
# ============================================================

wer_metric = load("wer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids    = np.argmax(pred_logits, axis=-1)
    pred_str    = processor.batch_decode(pred_ids)

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

# ============================================================
# 15  Trainer
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=processor,
    compute_metrics=compute_metrics,
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/mms-300m
Key                          | Status     | 
-----------------------------+------------+-
project_q.weight             | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: facebook/mms-300m
Total parameters: 315,468,445


model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

In [ ]:
# ============================================================
# 16  Train
# ============================================================

print("\n========== STARTING TRAINING ==========")
print(f"Model                : {MODEL_NAME}")
print(f"Transformers version : {transformers.__version__}")
print(f"Nyanja vocabulary     : {len(processor.tokenizer)} characters")
print(f"Train samples        : {len(train_dataset)}")
print(f"Val samples          : {len(val_dataset)}")
print(f"Epochs               : {training_args.num_train_epochs}")
print(f"Effective batch size : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate        : {training_args.learning_rate}")
print("========================================\n")

trainer.train()

# ============================================================
# 17  Save Model & Processor
# ============================================================

model.save_pretrained(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
print(f"Model and processor saved to {FINAL_DIR}")

# ============================================================
# 18  Final Evaluation
# ============================================================

print("\n========== FINAL EVALUATION ==========")
eval_results = trainer.evaluate()
print(f"Final WER: {eval_results['eval_wer']:.4f}  ({eval_results['eval_wer']*100:.2f}%)")

# ============================================================
# 19  Test Inference  (spot-check 5 validation samples)
# ============================================================

def transcribe(idx):
    example      = val_dataset[idx]
    device       = model.device
    input_values = torch.tensor(example["input_values"]).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_values).logits

    pred_ids   = torch.argmax(logits, dim=-1)
    prediction = processor.decode(pred_ids[0])

    label_ids = [i for i in example["labels"] if i != -100]
    reference = processor.decode(label_ids)

    print(f"\n--- Sample {idx} ---")
    print("REFERENCE :", reference)
    print("PREDICTION:", prediction)

print("\n========== TEST INFERENCE ==========")
for i in range(5):
    transcribe(i)

print("\n========== TRAINING PIPELINE COMPLETE ==========")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 28, 'bos_token_id': 27}.



========== STARTING TRAINING ==========
Model                : facebook/mms-300m
Transformers version : 5.0.0
Nyanja vocabulary     : 29 characters
Train samples        : 7290
Val samples          : 811
Epochs               : 5
Effective batch size : 8
Learning rate        : 0.0001



Step,Training Loss,Validation Loss,Wer
200,7.091002,3.016788,0.994745
400,5.776262,2.867805,0.994745
600,1.980693,0.504807,0.536338
800,0.901096,0.277201,0.358033
1000,0.735755,0.222972,0.306288
1200,0.656285,0.192664,0.285447
1400,0.585521,0.177014,0.265052
1600,0.520691,0.153505,0.253652
1800,0.478062,0.147744,0.241272
2000,0.453948,0.146571,0.230584


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and processor saved to ./nyamja-mms-final

========== FINAL EVALUATION ==========


Final WER: 0.1964  (19.64%)

========== TEST INFERENCE ==========

--- Sample 0 ---
REFERENCE : mnyamata mu kabudula was bulu ajumpa pa bedi
PREDICTION: mnyamata mu kabudula wa buluu ajumpa pa bedi

--- Sample 1 ---
REFERENCE : mwamuna wina akuyang ana atatambasula manja ndi chala pamene wina akuyang ana kumbuyo kwake
PREDICTION: mwamuna wina akuyang ana atatambasula manjandi chala pamene wina akuyang ana kumbuyo kwake

--- Sample 2 ---
REFERENCE : bambo wina wovala mathanji osambira akungoima uku akuthamanga pamchenga pamene munthu wovala malaya oyera kumbuyo amawotchi
PREDICTION: bambo wina wovala mathanji osambira akugoima uku akuthamanga pamchenga pamene munthu wovala malaya oyera kumbuyo amawotchi

--- Sample 3 ---
REFERENCE : bambo wina akuponya mpira pabwalo louma ndi mitambo kumbuyo kumbuyo
PREDICTION: bambo wina akuponya mpira pabwalo louma ndi mitambo kumbuyo kumbuyo

--- Sample 4 ---
REFERENCE : anthu awiri ovala zipewa atakwera mawilo anayi pafupi ndi nyanja
PREDICTION: ant